# 04 — Clean 2-Agent Colab Prototype (`Agent 1 -> Agent 2`)

Loads `Qwen2.5-3B-Instruct` once in 4-bit, attaches both `qwen2.5-3b-cc-agent1` and `qwen2.5-3b-cc-agent2` LoRA adapters, safely parses JSON outputs, and exposes a Gradio prototype.

*(The full local multi-agent system with 9-role interviews, quality/conflict detection, compliance & risk registers, RBAC, audit trail, evaluation, and SRS/RTM export is in `../ui/app.py`.)*

In [ ]:
!pip install -q gradio
!unzip -q -o qwen2.5-3b-cc-agent1.zip -d .
!unzip -q -o qwen2.5-3b-cc-agent2.zip -d .

In [ ]:
import re

def extract_json_clean(raw_text: str):
    """Strip optional prose prefixes or markdown fences before json.loads()."""
    text = raw_text.strip()
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text, re.I)
    if fence:
        text = fence.group(1).strip()
    starts = [i for i in (text.find("["), text.find("{")) if i != -1]
    if not starts:
        raise ValueError("No JSON payload found in model output.")
    s = min(starts)
    e = text.rfind("]" if text[s] == "[" else "}")
    return json.loads(text[s : e + 1])

import gradio as gr
import json
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print("Loading Base Model... (This takes a few seconds)")
# 1. Load the Base Model once
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print("Loading Your Trained Agents...")
# 2. Attach both of your trained agents to the base model
model.load_adapter("qwen2.5-3b-cc-agent1", adapter_name="agent1")
model.load_adapter("qwen2.5-3b-cc-agent2", adapter_name="agent2")
print("System Ready!")

# 3. The function that runs both agents automatically
def process_pipeline(functionality, context):
    try:
        # --- Run Agent 1 ---
        model.set_adapter("agent1")
        agent1_messages = [
            {"role": "system", "content": "You are a requirements engineering assistant for IT Security. Given a functionality and Common Criteria context, output a JSON array of requirements with specific security reasoning."},
            {"role": "user", "content": f"Functionality: {functionality}\n\nRetrieved Context:\n{context}"}
        ]
        inputs1 = tokenizer.apply_chat_template(agent1_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
        outputs1 = model.generate(input_ids=inputs1, max_new_tokens=1024, use_cache=True)
        agent1_json_str = tokenizer.batch_decode(outputs1[:, inputs1.shape[1]:], skip_special_tokens=True)[0]

        # Format Agent 1 Output
        reqs = extract_json_clean(agent1_json_str)
        reqs_markdown = "### 📋 Identified Security Requirements\n\n"
        for r in reqs:
            reqs_markdown += f"**{r.get('req_id', '')}** ({r.get('category', '')})\n"
            reqs_markdown += f"- **Statement:** {r.get('statement', '')}\n"
            reqs_markdown += f"- **Reasoning:** _{r.get('reasoning', '')}_\n\n"

        # --- Run Agent 2 ---
        model.set_adapter("agent2")
        agent2_messages = [
            {"role": "system", "content": "You are a Secure SDLC Architect. Evaluate the provided requirements, assign a weight/pull factor to each to determine the dominant constraint, and output a JSON object recommending the optimal SDLC with justification."},
            {"role": "user", "content": f"Requirements:\n{agent1_json_str}"}
        ]
        inputs2 = tokenizer.apply_chat_template(agent2_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
        outputs2 = model.generate(input_ids=inputs2, max_new_tokens=1024, use_cache=True)
        agent2_json_str = tokenizer.batch_decode(outputs2[:, inputs2.shape[1]:], skip_special_tokens=True)[0]

        # Format Agent 2 Output
        sdlc = extract_json_clean(agent2_json_str)
        sdlc_markdown = f"### 🏗️ Recommended SDLC: **{sdlc.get('recommended_sdlc', '')}**\n\n"
        sdlc_markdown += f"**Dominant Constraint:** {sdlc.get('dominant_constraint', '')}\n\n"
        sdlc_markdown += f"**Justification:** {sdlc.get('justification', '')}\n\n"

        return reqs_markdown, sdlc_markdown

    except Exception as e:
        return f"**Error parsing output. Ensure context is valid.**\n{str(e)}", "Error during Agent 2 execution."

# 4. Build the Web Interface
with gr.Blocks() as demo:
    gr.Markdown("# 🛡️ Multi-Agent Automated Security Engineering Tool")
    gr.Markdown("Enter a system functionality and its Common Criteria (CC) context. Agent 1 will generate requirements, and Agent 2 will recommend the optimal SDLC.")

    with gr.Row():
        with gr.Column():
            func_input = gr.Textbox(label="System Functionality", placeholder="e.g., Secure Firmware Update", lines=1)
            ctx_input = gr.Textbox(label="Common Criteria Context", placeholder="Paste raw CC SFRs here...", lines=5)
            submit_btn = gr.Button("🚀 Run End-to-End Analysis", variant="primary")

            # Add a quick example you can click during the demo
            gr.Examples(
                examples=[
                    ["Intrusion Prevention System (IPS)", "- IPS_ABD_EXT.1.1: The TSF shall support definition of expected throughput and anomaly thresholds.\n- IPS_EXT.1.4: In inline mode: block/drop the traffic flow in real time."]
                ],
                inputs=[func_input, ctx_input]
            )

        with gr.Column():
            agent1_output = gr.Markdown(label="Agent 1: Security Requirements")
            gr.Markdown("---")
            agent2_output = gr.Markdown(label="Agent 2: SDLC Recommendation")

    submit_btn.click(fn=process_pipeline, inputs=[func_input, ctx_input], outputs=[agent1_output, agent2_output])

# 5. Launch the App!
demo.launch(share=True)